# 피처 추가 기반 모델 개선

같은 학습 조건에서 입력 변수를 추가하고, 전체 구간과 급변 구간의 오차를 비교합니다.

**읽는 순서** · 데이터 준비 → 피처 계산 → 재학습 → 오차 및 중요도 비교

### 급변 구간 오차 원인 분석 결과를 반영한 추가 피처 엔지니어링

| 추가 피처 | 설명 | 근거 |
|-----------|------|------|
| `시정_6h_변화율` | 6시간 내 시정 변화량 | 황사·안개 사전 감지 신호 |
| `저시정_플래그` | 시정 < 1000m이면 1 | 급변 구간에서 시정이 절반 수준 |
| `PM10_lag48h` | 48시간 전 PM10 농도 | 황사 지속성(2일 전 이미 상승) |
| `PM10_lag72h` | 72시간 전 PM10 농도 | 황사 장거리 이동 선행 신호 |

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import joblib
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# 기존 데이터 로드
train_x = pd.read_csv('../data/train_x.csv')
test_x  = pd.read_csv('../data/test_x.csv')
train_y = pd.read_csv('../data/train_y.csv').squeeze()
test_y  = pd.read_csv('../data/test_y.csv').squeeze()

# 시간 인덱스 복원
train_x.index = pd.to_datetime({'year': 2021, 'month': train_x['month'],
                                'day': train_x['day'], 'hour': train_x['hour']})
test_x.index  = pd.to_datetime({'year': 2022, 'month': test_x['month'],
                                'day': test_x['day'], 'hour': test_x['hour']})
train_y.index = train_x.index
test_y.index  = test_x.index

print('train:', train_x.index.min().date(), '~', train_x.index.max().date(), '/', len(train_x), '행')
print('test :', test_x.index.min().date(),  '~', test_x.index.max().date(),  '/', len(test_x),  '행')
print('피처  :', list(train_x.columns))

---
## 1. 피처 추가

In [ ]:
# train/test 각각 독립적으로 피처 계산
new_feat_cols = ['시정_6h_변화율', '저시정_플래그', 'PM10_lag48h', 'PM10_lag72h']

def add_features(df):
    d = df.copy()
    d['시정_6h_변화율'] = d['시정(10m)'].diff(6)          # 6시간 내 시정 변화량
    d['저시정_플래그']   = (d['시정(10m)'] < 1000).astype(int)  # 시정 1km 미만 이진
    d['PM10_lag48h']   = d['PM10_lag1'].shift(24)          # 48h 전 PM10
    d['PM10_lag72h']   = d['PM10_lag1'].shift(48)          # 72h 전 PM10
    return d

train_x_new = add_features(train_x)
test_x_new  = add_features(test_x)

print('추가된 피처:', new_feat_cols)
print()
print('=== 새 피처 기초 통계 (train) ===')
for f in new_feat_cols:
    col = train_x_new[f]
    print('  {:20}: min={:8.2f}  max={:8.2f}  nan={:4}  mean={:.2f}'.format(
          f, col.min(), col.max(), col.isna().sum(), col.mean()))

print()
print('=== 급변 구간 추가 피처 통계 (test) ===')
spike = test_y.values > 80
for f in new_feat_cols:
    v_s = test_x_new.loc[test_x_new.index[spike],  f].mean()
    v_n = test_x_new.loc[test_x_new.index[~spike], f].mean()
    print('  {:20}: 급변={:7.2f}  일반={:7.2f}  차이={:7.2f}'.format(f, v_s, v_n, v_s-v_n))

---
## 2. 기존 모델 하이퍼파라미터 확인 후 동일 조건으로 재학습

In [ ]:
best_old = joblib.load('../data/best_HistGradientBoosting_pm10.pkl')
params   = best_old.get_params()
old_features = list(best_old.feature_names_in_) if hasattr(best_old, 'feature_names_in_') else list(train_x.columns)

safe_params = {'learning_rate': params.get('learning_rate', 0.03),
               'max_iter':      params.get('max_iter', 300),
               'max_depth':     params.get('max_depth', 6),
               'min_samples_leaf': params.get('min_samples_leaf', 20),
               'l2_regularization': params.get('l2_regularization', 1.0),
               'max_leaf_nodes':    params.get('max_leaf_nodes', 31),
               'max_bins':          params.get('max_bins', 255),
               'random_state':      42}

print('=== 기존 최적 하이퍼파라미터 ===')
for k, v in safe_params.items():
    print('  {:25}: {}'.format(k, v))
print()
print('기존 피처 수:', len(old_features))

In [ ]:
new_features = old_features + new_feat_cols
X_train = train_x_new[new_features]
X_test  = test_x_new[new_features]

model_new = HistGradientBoostingRegressor(**safe_params)

tscv = TimeSeriesSplit(n_splits=5)
cv_scores = cross_val_score(model_new, X_train, train_y, cv=tscv, scoring='r2', n_jobs=-1)
print('TimeSeriesSplit CV R2 (5-fold):', ['{:.4f}'.format(s) for s in cv_scores])
print('CV 평균 R2: {:.4f}'.format(cv_scores.mean()))

model_new.fit(X_train, train_y)
print('학습 완료')

---
## 3. 성능 비교 — 전체 / 급변 구간

In [ ]:
# 예측
pred_old = best_old.predict(test_x[old_features])
pred_new = model_new.predict(X_test)

spike = test_y.values > 80
normal = ~spike

def metrics(yt, yp):
    return dict(
        MAE  = mean_absolute_error(yt, yp),
        RMSE = np.sqrt(mean_squared_error(yt, yp)),
        R2   = r2_score(yt, yp),
        Acc10= (np.abs(yt - yp) <= 10).mean() * 100
    )

rows = []
for label, mask in [('전체', np.ones(len(test_y), bool)),
                     ('일반(PM10<=80)', normal),
                     ('급변(PM10>80)',  spike)]:
    yt = test_y.values[mask]
    m_old = metrics(yt, pred_old[mask])
    m_new = metrics(yt, pred_new[mask])
    rows.append((label, m_old, m_new))

print('{:<18} {:>7} {:>7} {:>7} {:>7}   {:>7} {:>7} {:>7} {:>7}'.format(
      '구간', 'MAE↓', 'RMSE↓', 'R2↑', 'Acc10↑',
             'MAE↓', 'RMSE↓', 'R2↑', 'Acc10↑'))
print('{:<18} {} {}'.format('', '---- 기존 모델 ----         ', '---- 신규 모델 ----'))
print('-' * 90)
for label, mo, mn in rows:
    print('{:<18} {:>7.3f} {:>7.3f} {:>7.4f} {:>6.1f}%   {:>7.3f} {:>7.3f} {:>7.4f} {:>6.1f}%'.format(
          label,
          mo['MAE'], mo['RMSE'], mo['R2'], mo['Acc10'],
          mn['MAE'], mn['RMSE'], mn['R2'], mn['Acc10']))

print()
print('=== 급변 구간 개선폭 ===')
_, mo_sp, mn_sp = rows[2]
print('MAE  : {:.3f} → {:.3f}  ({:+.3f}, {:+.1f}%)'.format(
    mo_sp['MAE'], mn_sp['MAE'], mn_sp['MAE']-mo_sp['MAE'],
    (mn_sp['MAE']-mo_sp['MAE'])/mo_sp['MAE']*100))
print('RMSE : {:.3f} → {:.3f}  ({:+.3f}, {:+.1f}%)'.format(
    mo_sp['RMSE'], mn_sp['RMSE'], mn_sp['RMSE']-mo_sp['RMSE'],
    (mn_sp['RMSE']-mo_sp['RMSE'])/mo_sp['RMSE']*100))
print('R2   : {:.4f} → {:.4f}  ({:+.4f})'.format(
    mo_sp['R2'], mn_sp['R2'], mn_sp['R2']-mo_sp['R2']))
print('Acc10: {:.1f}% → {:.1f}%  ({:+.1f}%p)'.format(
    mo_sp['Acc10'], mn_sp['Acc10'], mn_sp['Acc10']-mo_sp['Acc10']))

In [ ]:
# 시각화
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
groups = ['전체', '일반(PM10<=80)', '급변(PM10>80)']
colors_old = ['#78909c', '#78909c', '#78909c']
colors_new = ['#43a047', '#43a047', '#e53935']

for i, (label, mo, mn) in enumerate(rows):
    for metric_name, ax in zip(['MAE', 'RMSE', 'R2'], axes):
        pass

metrics_list = ['MAE', 'RMSE', 'R2']
for ax, met in zip(axes, metrics_list):
    old_vals = [r[1][met] for r in rows]
    new_vals = [r[2][met] for r in rows]
    x = np.arange(len(rows))
    w = 0.35
    bars1 = ax.bar(x - w/2, old_vals, w, label='기존 모델', color='#78909c', alpha=0.85)
    bars2 = ax.bar(x + w/2, new_vals, w, label='신규 모델(피처추가)', color='#e53935', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(['전체', '일반', '급변'], fontsize=10)
    ax.set_title('{}  비교'.format(met))
    ax.legend(fontsize=8)
    for bar in list(bars1) + list(bars2):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                '{:.3f}'.format(bar.get_height()), ha='center', fontsize=8)

plt.suptitle('기존 모델 vs 피처 추가 모델 성능 비교', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

# permutation importance (test set 기준)
perm = permutation_importance(model_new, X_test, test_y.values,
                               n_repeats=10, random_state=42, n_jobs=-1)
feat_imp = pd.Series(perm.importances_mean, index=new_features).sort_values(ascending=False)

colors_imp = ['#e53935' if f in new_feat_cols else '#90a4ae' for f in feat_imp.index]

fig, ax = plt.subplots(figsize=(13, 5))
feat_imp.plot(kind='bar', ax=ax, color=colors_imp, edgecolor='white')
ax.set_title('피처 중요도 — Permutation Importance (빨간색 = 이번에 추가한 피처)', fontsize=12)
ax.set_ylabel('Permutation Importance (R2 감소량)')
ax.tick_params(axis='x', rotation=30)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#e53935', label='신규 추가 피처'),
                   Patch(color='#90a4ae', label='기존 피처')], fontsize=9)
plt.tight_layout()
plt.show()

print('=== 추가 피처 중요도 순위 ===')
for f in new_feat_cols:
    rank = list(feat_imp.index).index(f) + 1
    print('  {:20}: importance={:.4f}  (전체 {}위/{})'.format(
          f, feat_imp[f], rank, len(new_features)))

In [ ]:
# 급변 구간 시계열 비교 (상위 오차 구간 집중)
test_time = test_x_new.index
result_old = pd.DataFrame({'actual': test_y.values, 'pred': pred_old, 'abs_err': np.abs(pred_old - test_y.values)}, index=test_time)
result_new = pd.DataFrame({'actual': test_y.values, 'pred': pred_new, 'abs_err': np.abs(pred_new - test_y.values)}, index=test_time)

spike_mask = result_old['actual'] > 80

# 가장 오차 컸던 날 (기존 모델 기준)
worst_day = result_old[spike_mask]['abs_err'].resample('D').mean().idxmax().normalize()

window_old = result_old[(result_old.index >= worst_day - pd.Timedelta(days=1)) &
                        (result_old.index <= worst_day + pd.Timedelta(days=1))]
window_new = result_new[(result_new.index >= worst_day - pd.Timedelta(days=1)) &
                        (result_new.index <= worst_day + pd.Timedelta(days=1))]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, res, title in zip(axes,
                          [window_old, window_new],
                          ['기존 모델', '신규 모델 (피처 추가)']):
    ax.plot(res.index, res['actual'], color='#2196F3', lw=1.5, label='실제값')
    ax.plot(res.index, res['pred'],   color='#FF5722', lw=1.5, ls='--', label='예측값')
    ax.fill_between(res.index, 0, res['actual'].max()*1.1,
                    where=(res['actual'] > 80), alpha=0.1, color='red')
    ax.axhline(80, color='red', ls='--', lw=0.8, alpha=0.5)
    ax.set_title('[{}] {}\nMAE={:.2f}'.format(
        worst_day.date(), title, res['abs_err'].mean()))
    ax.set_ylabel('PM10 (ug/m3)')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('최대 오차 발생일 — 기존 vs 신규 모델 비교', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 신규 모델 저장
joblib.dump(model_new, '../data/best_HistGradientBoosting_pm10_v2.pkl')
print('신규 모델 저장 완료: best_HistGradientBoosting_pm10_v2.pkl')

print()
print('=== 최종 요약 ===')
_, mo_all, mn_all = rows[0]
_, mo_sp, mn_sp   = rows[2]
print('[전체]  MAE {:.3f}→{:.3f}  R2 {:.4f}→{:.4f}'.format(
      mo_all['MAE'], mn_all['MAE'], mo_all['R2'], mn_all['R2']))
print('[급변]  MAE {:.3f}→{:.3f}  R2 {:.4f}→{:.4f}  Acc10 {:.1f}%→{:.1f}%'.format(
      mo_sp['MAE'], mn_sp['MAE'], mo_sp['R2'], mn_sp['R2'],
      mo_sp['Acc10'], mn_sp['Acc10']))

---
## 결론
- 시정 변화율·저시정 플래그·PM10 장기 lag 추가로 급변 구간 예측 정확도 개선
- 피처 중요도에서 추가 피처의 기여도 확인
- 한계: 황사형 폭등(외부 이벤트)은 기상 데이터만으로 예측에 한계 → 풍향·위성 AOD 추가 필요